In [1]:
from pathlib import Path
import sqlite3
import pandas as pd

# Carpeta del Día 8.
CARPETA_DATOS = Path("datos_dia_08")
CARPETA_DATOS.mkdir(exist_ok=True)

# Base de datos del Día 8.
RUTA_DB = CARPETA_DATOS / "ventas_normalizacion.db"

print("Carpeta de trabajo:")
print(CARPETA_DATOS.resolve())

print("\nBase de datos:")
print(RUTA_DB.resolve())

Carpeta de trabajo:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_08

Base de datos:
C:\Users\alumn\OneDrive\Escritorio\Cursos 2026-02\Persistencia de datos con python\Códigos\notebooks\datos_dia_08\ventas_normalizacion.db


In [2]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS ventas_no_normalizadas")

cursor.execute("""
CREATE TABLE ventas_no_normalizadas (
    id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    fecha TEXT NOT NULL,

    cliente_nombre TEXT NOT NULL,
    cliente_correo TEXT NOT NULL,
    cliente_ciudad TEXT NOT NULL,

    productos TEXT NOT NULL,
    cantidades TEXT NOT NULL,
    precios_unitarios TEXT NOT NULL
)
""")

conexion.commit()
conexion.close()

print("Tabla no normalizada creada correctamente.")

Tabla no normalizada creada correctamente.


In [3]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

datos = [
    (
        1,
        "2026-06-20",
        "Ana López",
        "ana@example.com",
        "CDMX",
        "Mouse inalámbrico, Teclado mecánico",
        "2, 1",
        "249.90, 899.00"
    ),
    (
        2,
        "2026-06-21",
        "Carlos Pérez",
        "carlos@example.com",
        "Guadalajara",
        "Monitor 24 pulgadas",
        "1",
        "3299.00"
    ),
    (
        3,
        "2026-06-21",
        "Ana López",
        "ana@example.com",
        "CDMX",
        "Cable HDMI, Memoria USB 64GB",
        "3, 2",
        "120.00, 150.00"
    )
]

cursor.executemany("""
INSERT INTO ventas_no_normalizadas (
    id_venta,
    fecha,
    cliente_nombre,
    cliente_correo,
    cliente_ciudad,
    productos,
    cantidades,
    precios_unitarios
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", datos)

conexion.commit()
conexion.close()

print("Registros no normalizados insertados correctamente.")

Registros no normalizados insertados correctamente.


In [4]:
conexion = sqlite3.connect(RUTA_DB)

tabla_no_normalizada = pd.read_sql_query("""
SELECT *
FROM ventas_no_normalizadas
ORDER BY id_registro
""", conexion)

conexion.close()

tabla_no_normalizada

,id_registro,id_venta,fecha,cliente_nombre,cliente_correo,cliente_ciudad,productos,cantidades,precios_unitarios
0,1,1,2026-06-20,Ana López,ana@example.com,CDMX,"Mouse inalámbrico, Teclado mecánico","2, 1","249.90, 899.00"
1,2,2,2026-06-21,Carlos Pérez,carlos@example.com,Guadalajara,Monitor 24 pulgadas,1,3299.00
2,3,3,2026-06-21,Ana López,ana@example.com,CDMX,"Cable HDMI, Memoria USB 64GB","3, 2","120.00, 150.00"


In [5]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS ventas_1fn")

cursor.execute("""
CREATE TABLE ventas_1fn (
    id_registro INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    fecha TEXT NOT NULL,

    cliente_nombre TEXT NOT NULL,
    cliente_correo TEXT NOT NULL,
    cliente_ciudad TEXT NOT NULL,

    producto_nombre TEXT NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0),
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0)
)
""")

conexion.commit()
conexion.close()

print("Tabla ventas_1fn creada correctamente.")

Tabla ventas_1fn creada correctamente.


In [6]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

ventas_1fn = [
    (1, "2026-06-20", "Ana López", "ana@example.com", "CDMX", "Mouse inalámbrico", 2, 249.90),
    (1, "2026-06-20", "Ana López", "ana@example.com", "CDMX", "Teclado mecánico", 1, 899.00),
    (2, "2026-06-21", "Carlos Pérez", "carlos@example.com", "Guadalajara", "Monitor 24 pulgadas", 1, 3299.00),
    (3, "2026-06-21", "Ana López", "ana@example.com", "CDMX", "Cable HDMI", 3, 120.00),
    (3, "2026-06-21", "Ana López", "ana@example.com", "CDMX", "Memoria USB 64GB", 2, 150.00)
]

cursor.executemany("""
INSERT INTO ventas_1fn (
    id_venta,
    fecha,
    cliente_nombre,
    cliente_correo,
    cliente_ciudad,
    producto_nombre,
    cantidad,
    precio_unitario
)
VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", ventas_1fn)

conexion.commit()
conexion.close()

print("Datos insertados en ventas_1fn.")

Datos insertados en ventas_1fn.


In [7]:
conexion = sqlite3.connect(RUTA_DB)

tabla_1fn = pd.read_sql_query("""
SELECT
    id_registro,
    id_venta,
    fecha,
    cliente_nombre,
    cliente_correo,
    cliente_ciudad,
    producto_nombre,
    cantidad,
    precio_unitario,
    cantidad * precio_unitario AS subtotal
FROM ventas_1fn
ORDER BY id_venta, id_registro
""", conexion)

conexion.close()

tabla_1fn

,id_registro,id_venta,fecha,cliente_nombre,cliente_correo,cliente_ciudad,producto_nombre,cantidad,precio_unitario,subtotal
0,1,1,2026-06-20,Ana López,ana@example.com,CDMX,Mouse inalámbrico,2,249.9,499.8
1,2,1,2026-06-20,Ana López,ana@example.com,CDMX,Teclado mecánico,1,899.0,899.0
2,3,2,2026-06-21,Carlos Pérez,carlos@example.com,Guadalajara,Monitor 24 pulgadas,1,3299.0,3299.0
3,4,3,2026-06-21,Ana López,ana@example.com,CDMX,Cable HDMI,3,120.0,360.0
4,5,3,2026-06-21,Ana López,ana@example.com,CDMX,Memoria USB 64GB,2,150.0,300.0


In [8]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS detalle_ventas_2fn")
cursor.execute("DROP TABLE IF EXISTS ventas_2fn")
cursor.execute("DROP TABLE IF EXISTS productos_2fn")
cursor.execute("DROP TABLE IF EXISTS clientes_2fn")

cursor.execute("""
CREATE TABLE clientes_2fn (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    ciudad TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE productos_2fn (
    id_producto INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL UNIQUE,
    precio_unitario REAL NOT NULL CHECK(precio_unitario > 0)
)
""")

cursor.execute("""
CREATE TABLE ventas_2fn (
    id_venta INTEGER PRIMARY KEY,
    fecha TEXT NOT NULL,
    id_cliente INTEGER NOT NULL
)
""")

cursor.execute("""
CREATE TABLE detalle_ventas_2fn (
    id_detalle INTEGER PRIMARY KEY AUTOINCREMENT,
    id_venta INTEGER NOT NULL,
    id_producto INTEGER NOT NULL,
    cantidad INTEGER NOT NULL CHECK(cantidad > 0)
)
""")

conexion.commit()
conexion.close()

print("Tablas para 2FN creadas correctamente.")

Tablas para 2FN creadas correctamente.


In [9]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

clientes = [
    ("Ana López", "ana@example.com", "CDMX"),
    ("Carlos Pérez", "carlos@example.com", "Guadalajara")
]

productos = [
    ("Mouse inalámbrico", 249.90),
    ("Teclado mecánico", 899.00),
    ("Monitor 24 pulgadas", 3299.00),
    ("Cable HDMI", 120.00),
    ("Memoria USB 64GB", 150.00)
]

ventas = [
    (1, "2026-06-20", 1),
    (2, "2026-06-21", 2),
    (3, "2026-06-21", 1)
]

detalle_ventas = [
    (1, 1, 2),
    (1, 2, 1),
    (2, 3, 1),
    (3, 4, 3),
    (3, 5, 2)
]

cursor.executemany("""
INSERT INTO clientes_2fn (nombre, correo, ciudad)
VALUES (?, ?, ?)
""", clientes)

cursor.executemany("""
INSERT INTO productos_2fn (nombre, precio_unitario)
VALUES (?, ?)
""", productos)

cursor.executemany("""
INSERT INTO ventas_2fn (id_venta, fecha, id_cliente)
VALUES (?, ?, ?)
""", ventas)

cursor.executemany("""
INSERT INTO detalle_ventas_2fn (id_venta, id_producto, cantidad)
VALUES (?, ?, ?)
""", detalle_ventas)

conexion.commit()
conexion.close()

print("Datos insertados en tablas 2FN.")

Datos insertados en tablas 2FN.


In [10]:
conexion = sqlite3.connect(RUTA_DB)

print("Clientes 2FN")
display(pd.read_sql_query("SELECT * FROM clientes_2fn", conexion))

print("Productos 2FN")
display(pd.read_sql_query("SELECT * FROM productos_2fn", conexion))

print("Ventas 2FN")
display(pd.read_sql_query("SELECT * FROM ventas_2fn", conexion))

print("Detalle ventas 2FN")
display(pd.read_sql_query("SELECT * FROM detalle_ventas_2fn", conexion))

conexion.close()

Clientes 2FN


,id_cliente,nombre,correo,ciudad
0,1,Ana López,ana@example.com,CDMX
1,2,Carlos Pérez,carlos@example.com,Guadalajara


Productos 2FN


,id_producto,nombre,precio_unitario
0,1,Mouse inalámbrico,249.9
1,2,Teclado mecánico,899.0
2,3,Monitor 24 pulgadas,3299.0
3,4,Cable HDMI,120.0
4,5,Memoria USB 64GB,150.0


Ventas 2FN


,id_venta,fecha,id_cliente
0,1,2026-06-20,1
1,2,2026-06-21,2
2,3,2026-06-21,1


Detalle ventas 2FN


,id_detalle,id_venta,id_producto,cantidad
0,1,1,1,2
1,2,1,2,1
2,3,2,3,1
3,4,3,4,3
4,5,3,5,2


In [11]:
conexion = sqlite3.connect(RUTA_DB)

venta_completa_2fn = pd.read_sql_query("""
SELECT
    v.id_venta,
    v.fecha,
    c.nombre AS cliente,
    c.correo,
    p.nombre AS producto,
    dv.cantidad,
    p.precio_unitario,
    dv.cantidad * p.precio_unitario AS subtotal
FROM detalle_ventas_2fn dv
INNER JOIN ventas_2fn v
    ON dv.id_venta = v.id_venta
INNER JOIN clientes_2fn c
    ON v.id_cliente = c.id_cliente
INNER JOIN productos_2fn p
    ON dv.id_producto = p.id_producto
ORDER BY v.id_venta, dv.id_detalle
""", conexion)

conexion.close()

venta_completa_2fn

,id_venta,fecha,cliente,correo,producto,cantidad,precio_unitario,subtotal
0,1,2026-06-20,Ana López,ana@example.com,Mouse inalámbrico,2,249.9,499.8
1,1,2026-06-20,Ana López,ana@example.com,Teclado mecánico,1,899.0,899.0
2,2,2026-06-21,Carlos Pérez,carlos@example.com,Monitor 24 pulgadas,1,3299.0,3299.0
3,3,2026-06-21,Ana López,ana@example.com,Cable HDMI,3,120.0,360.0
4,3,2026-06-21,Ana López,ana@example.com,Memoria USB 64GB,2,150.0,300.0


In [12]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

cursor.execute("DROP TABLE IF EXISTS clientes_3fn")
cursor.execute("DROP TABLE IF EXISTS ciudades_3fn")

cursor.execute("""
CREATE TABLE ciudades_3fn (
    id_ciudad INTEGER PRIMARY KEY AUTOINCREMENT,
    ciudad TEXT NOT NULL UNIQUE,
    estado TEXT NOT NULL
)
""")

cursor.execute("""
CREATE TABLE clientes_3fn (
    id_cliente INTEGER PRIMARY KEY AUTOINCREMENT,
    nombre TEXT NOT NULL,
    correo TEXT NOT NULL UNIQUE,
    id_ciudad INTEGER NOT NULL
)
""")

conexion.commit()
conexion.close()

print("Tablas para 3FN creadas correctamente.")

Tablas para 3FN creadas correctamente.


In [15]:
conexion = sqlite3.connect(RUTA_DB)
cursor = conexion.cursor()

ciudades = [
    ("CDMX", "Ciudad de México"),
    ("Guadalajara", "Jalisco"),
    ("Monterrey", "Nuevo León")
]

clientes_3fn = [
    ("Ana López", "ana@example.com", 1),
    ("Carlos Pérez", "carlos@example.com", 2),
    ("María Torres", "maria@example.com", 3)
]

cursor.executemany("""
INSERT INTO ciudades_3fn (ciudad, estado)
VALUES (?, ?)
""", ciudades)

cursor.executemany("""
INSERT INTO clientes_3fn (nombre, correo, id_ciudad)
VALUES (?, ?, ?)
""", clientes_3fn)

conexion.commit()
conexion.close()

print("Datos insertados en tablas 3FN.")

Datos insertados en tablas 3FN.


In [16]:
conexion = sqlite3.connect(RUTA_DB)

clientes_con_ubicacion = pd.read_sql_query("""
SELECT
    c.id_cliente,
    c.nombre,
    c.correo,
    ci.ciudad,
    ci.estado
FROM clientes_3fn c
INNER JOIN ciudades_3fn ci
    ON c.id_ciudad = ci.id_ciudad
ORDER BY c.id_cliente
""", conexion)

conexion.close()

clientes_con_ubicacion

,id_cliente,nombre,correo,ciudad,estado
0,1,Ana López,ana@example.com,CDMX,Ciudad de México
1,2,Carlos Pérez,carlos@example.com,Guadalajara,Jalisco
2,3,María Torres,maria@example.com,Monterrey,Nuevo León


In [17]:
comparacion = pd.DataFrame([
    {
        "etapa": "No normalizada",
        "caracteristica": "Tiene listas dentro de una celda",
        "problema_resuelto": "Ninguno todavía"
    },
    {
        "etapa": "1FN",
        "caracteristica": "Cada celda tiene un solo valor",
        "problema_resuelto": "Elimina grupos repetidos dentro de celdas"
    },
    {
        "etapa": "2FN",
        "caracteristica": "Se separan datos que dependen de distintas llaves",
        "problema_resuelto": "Reduce redundancia de clientes, productos y ventas"
    },
    {
        "etapa": "3FN",
        "caracteristica": "Elimina dependencias transitivas",
        "problema_resuelto": "Evita datos que dependen de otros campos no llave"
    }
])

comparacion

,etapa,caracteristica,problema_resuelto
0,No normalizada,Tiene listas dentro de una celda,Ninguno todavía
1,1FN,Cada celda tiene un solo valor,Elimina grupos repetidos dentro de celdas
2,2FN,Se separan datos que dependen de distintas llaves,"Reduce redundancia de clientes, productos y ve..."
3,3FN,Elimina dependencias transitivas,Evita datos que dependen de otros campos no llave
